
# DCVLR Baseline Analysis Notebook

This Colab notebook aggregates DCVLR benchmark datasets, evaluates a baseline model, and finds similar training examples for unsolved problems.



## Setup
Install required packages. This cell is intended for Google Colab execution.


In [ ]:

!pip install -q oumi datasets transformers accelerate sentence-transformers faiss-cpu pillow


## Imports and Configuration

In [ ]:

from datasets import load_dataset, concatenate_datasets
from transformers import AutoProcessor, AutoModelForVision2Seq
from sentence_transformers import SentenceTransformer
from PIL import Image
import torch
import faiss
import json
from pathlib import Path



## Load Datasets
This section loads the curated DCVLR training dataset and benchmark datasets, then concatenates them for evaluation.


In [ ]:

benchmark_datasets = [
    "VMCBench_DEV",
    "OlympiadBench",
    "LiveXivVQA",
    "LiveXivTQA",
]

baseline_dataset = "penfever/multimodal-open-r1-8192-filtered-tighter"

image_column = "image"
question_column = "question"
answer_column = "answer"

loaded_datasets = []
for name in benchmark_datasets:
    ds = load_dataset(name, split="test", trust_remote_code=True)
    loaded_datasets.append(ds)

baseline_ds = load_dataset(baseline_dataset, split="train", trust_remote_code=True)
loaded_datasets.append(baseline_ds)

all_data = concatenate_datasets(loaded_datasets)
print(f"Total samples: {len(all_data)}")



## Run Inference with Qwen
This section runs the Qwen model on all samples. Additional models can be added by uncommenting the respective lines.


In [ ]:

model_name = "Qwen/Qwen2-VL-7B-Instruct"
# model_name = "ANOTHER/MODEL"  # Uncomment to try other baseline models

processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(model_name, device_map="auto", trust_remote_code=True)
model.eval()


In [ ]:

@torch.no_grad()
def generate_answer(sample):
    question = sample[question_column]
    image = sample[image_column]
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    inputs = processor(text=question, images=image, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=512)
    return processor.batch_decode(output, skip_special_tokens=True)[0]

predictions = []
for sample in all_data:
    try:
        pred = generate_answer(sample)
    except Exception as e:
        pred = f"ERROR: {e}"
    predictions.append(pred)

all_data = all_data.add_column("prediction", predictions)



## Evaluate and Filter Unsolved Problems
We mark a sample as solved when the predicted answer contains the ground truth answer as a substring (case-insensitive). Adjust the heuristic as needed.


In [ ]:

import re

def is_correct(pred, answer):
    if answer is None:
        return False
    return re.search(re.escape(str(answer)), str(pred), re.IGNORECASE) is not None

solved = [is_correct(p, a) for p, a in zip(all_data["prediction"], all_data[answer_column])]
all_data = all_data.add_column("solved", solved)

unsolved_data = all_data.filter(lambda x: not x["solved"])
print(f"Solved: {sum(solved)} / {len(solved)}")
print(f"Unsolved: {len(unsolved_data)}")

unsolved_path = Path("unsolved_examples.json")
unsolved_data.to_json(str(unsolved_path))
print(f"Saved unsolved samples to {unsolved_path}")



## Retrieve Similar Training Samples via Multimodal RAG
For each unsolved sample, we retrieve the most similar training examples using text and image embeddings.


In [ ]:

text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
clip_name = "openai/clip-vit-base-patch32"
clip_model = AutoModelForVision2Seq.from_pretrained(clip_name)
clip_processor = AutoProcessor.from_pretrained(clip_name)


def embed_sample(sample):
    text_emb = text_encoder.encode(sample[question_column], convert_to_tensor=True)
    image = sample[image_column]
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    clip_inputs = clip_processor(text=[sample[question_column]], images=image, return_tensors="pt")
    image_emb = clip_model.get_image_features(**clip_inputs)
    return torch.cat([text_emb, image_emb.squeeze(0)], dim=0)

train_embeddings = [embed_sample(sample) for sample in baseline_ds]
train_matrix = torch.stack(train_embeddings).cpu().numpy().astype("float32")
index = faiss.IndexFlatIP(train_matrix.shape[1])
index.add(train_matrix)

similar_examples = []
for sample in unsolved_data:
    query_emb = embed_sample(sample).unsqueeze(0).cpu().numpy().astype("float32")
    _, idxs = index.search(query_emb, k=5)
    similar_examples.append(idxs[0].tolist())

unsolved_data = unsolved_data.add_column("similar_training_idxs", similar_examples)
unsolved_with_similar_path = Path("unsolved_with_similar.json")
unsolved_data.to_json(str(unsolved_with_similar_path))
print(f"Saved unsolved samples with similar training indices to {unsolved_with_similar_path}")
